# PrintFarmEnv — Colab Quickstart

**What this notebook does (no GPU required):**
1. Installs `PrintFarmEnv` and its dependencies
2. Loads the environment via OpenEnv
3. Runs a naive-greedy baseline episode and prints the reward
4. Links to the full SFT and GRPO training notebooks

**This is the minimum-viable Colab artifact required by the organiser checklist.**
Full training is in `sft_dispatcher.ipynb` and `grpo_dispatcher.ipynb`.

---
**Swap model / task:** change the `TASK_ID` cell — everything adapts automatically.

## 0 · Install

In [ ]:
# ::: Code Generated by Copilot 1d9124a5-74d4-4de0-a709-859685f0dd8b. This comment will be removed automatically after the file is saved :::
import subprocess, sys, os

def run(cmd):
    subprocess.run(cmd, shell=True, check=True)

# Core env deps (no GPU needed for this quickstart)
run('pip install -q "openenv-core>=0.1" pydantic fastapi uvicorn')

# Clone repo (Colab only; skip locally)
IN_COLAB = 'google.colab' in str(get_ipython())
if IN_COLAB:
    if not os.path.exists('PrintFarmEnv'):
        run('git clone https://github.com/kshitizP/PrintFarmEnv.git')
    os.chdir('PrintFarmEnv')
    sys.path.insert(0, '.')

print('Setup complete.')

## 1 · Configuration

In [ ]:
# ── Change these to explore different tasks ───────────────────────────────────
TASK_ID  = "task_1"   # task_0_1 / task_0_2 / task_0_3 / task_1 / task_2 / task_3
SEED     = 42
EPISODES = 5          # quick demo; increase to 100 for stats

print(f"Task: {TASK_ID}  |  Seed: {SEED}  |  Episodes: {EPISODES}")

## 2 · Load the environment

In [ ]:
# ::: Code Generated by Copilot 1d9124a5-74d4-4de0-a709-859685f0dd8b. This comment will be removed automatically after the file is saved :::
from printfarm_env.env import PrintFarmEnvironment
from printfarm_env.models import FarmAction, FarmActionEnum

env = PrintFarmEnvironment()
obs = env.reset(seed=SEED, task_id=TASK_ID)

print(f"Environment loaded: {TASK_ID}")
print(f"  Printers   : {len(obs.printers)}")
print(f"  Jobs       : {len(obs.active_queue)}")
print(f"  Operators  : {len(obs.operators)}")
print(f"  Max steps  : {env.max_steps}")
print()
print("OpenEnv interface:")
print("  env.reset(seed, task_id)  → FarmObservation")
print("  env.step(FarmAction)      → FarmObservation  (reward in obs.metadata['step_reward_usd'])")
print("  env.state                 → current FarmObservation")

## 3 · Inspect one observation

In [ ]:
# ::: Code Generated by Copilot 1d9124a5-74d4-4de0-a709-859685f0dd8b. This comment will be removed automatically after the file is saved :::
import json

# Show first printer and first job
print("=== Printer 1 ===")
print(json.dumps(obs.printers[0].model_dump(), indent=2))
print()
print("=== First job ===")
print(json.dumps(obs.active_queue[0].model_dump(), indent=2))

## 4 · Run naive-greedy baseline

In [ ]:
# ::: Code Generated by Copilot 1d9124a5-74d4-4de0-a709-859685f0dd8b. This comment will be removed automatically after the file is saved :::
from baselines.naive_greedy import naive_action

def run_episode(task_id, seed, policy_fn, verbose=False):
    obs     = env.reset(seed=seed, task_id=task_id)
    steps   = 0
    total_r = 0.0
    while not obs.done:
        action  = policy_fn(obs)
        obs     = env.step(action)
        step_r  = obs.metadata.get("step_reward_usd", 0.0)
        total_r += step_r
        steps   += 1
        if verbose:
            print(f"  step {steps:3d}  action={action.action.value:<22s}  "
                  f"step_reward=${step_r:+.2f}  cumulative=${total_r:+.2f}")
    return obs.reward, total_r, steps

print(f"Running {EPISODES} naive-greedy episodes on {TASK_ID}...")
scores = []
for ep in range(EPISODES):
    score, profit, n_steps = run_episode(TASK_ID, SEED + ep, naive_action)
    scores.append(score)
    print(f"  ep={ep}  score={score:.4f}  profit=${profit:+.2f}  steps={n_steps}")

print()
print(f"Mean score : {sum(scores)/len(scores):.4f}")
print(f"Min  score : {min(scores):.4f}")
print(f"Max  score : {max(scores):.4f}")

## 5 · Reward breakdown per step (single episode, verbose)

In [ ]:
# ::: Code Generated by Copilot 1d9124a5-74d4-4de0-a709-859685f0dd8b. This comment will be removed automatically after the file is saved :::
obs     = env.reset(seed=SEED, task_id=TASK_ID)
records = []

while not obs.done:
    action = naive_action(obs)
    obs    = env.step(action)
    records.append({
        "step":      env.time_step,
        "action":    action.action.value,
        "step_r":    obs.metadata.get("step_reward_usd", 0.0),
        "breakdown": obs.reward_breakdown,
    })

# Print reward components table
print(f"{'step':>4}  {'action':<24} {'step_r':>8}  {'action':>8}  {'labor':>8}  {'physics':>8}  {'sla':>7}")
print("-" * 78)
for r in records[:20]:   # first 20 steps
    bd = r["breakdown"]
    print(f"{r['step']:>4}  {r['action']:<24} {r['step_r']:>+8.2f}  "
          f"{bd.get('action', 0):>+8.2f}  {bd.get('labor', 0):>+8.2f}  "
          f"{bd.get('physics', 0):>+8.2f}  {bd.get('sla', 0):>+7.2f}")

print()
print("Note: step_reward_usd is emitted PER STEP (dense reward signal).")
print("This is 'process-aware feedback' per Help Guide §9 / FAQ #11.")

## 6 · Plot reward trajectory

In [ ]:
import matplotlib.pyplot as plt

steps        = [r["step"]   for r in records]
step_rewards = [r["step_r"] for r in records]
cumulative   = [sum(step_rewards[:i+1]) for i in range(len(step_rewards))]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(steps, step_rewards, color=["steelblue" if v >= 0 else "salmon" for v in step_rewards])
ax1.axhline(0, color="black", linewidth=0.8)
ax1.set_xlabel("Step")
ax1.set_ylabel("step_reward_usd ($)")
ax1.set_title(f"Per-step reward — {TASK_ID} (naive-greedy)")

ax2.plot(steps, cumulative, color="steelblue")
ax2.axhline(0, color="black", linewidth=0.8)
ax2.set_xlabel("Step")
ax2.set_ylabel("Cumulative reward ($)")
ax2.set_title("Cumulative P&L")

plt.tight_layout()
plt.show()
print(f"Final episode score (grader): {records[-1]['step_r']:.4f}")

## 7 · What's next

| Notebook / Script | What it does |
|---|---|
| `scripts/build_sft_dataset.py` | Builds SFT dataset from clairvoyant + teacher trajectories |
| `notebooks/grpo_dispatcher.ipynb` | GRPO training with env P&L as the verifier reward |

**To train a model on this env in Colab:**
1. Open `notebooks/grpo_dispatcher.ipynb`
2. Set runtime → GPU (A100 recommended; T4 works for 3B models)
3. Change `MODEL_ID` if desired
4. Run all cells

**Full repo:** https://github.com/kshitizP/PrintFarmEnv

**Resources:**
- [OpenEnv docs](https://meta-pytorch.org/OpenEnv/)
- [TRL GRPO cookbook](https://huggingface.co/docs/trl)
- [Unsloth repo](https://github.com/unslothai/unsloth)